# Chapter 3 Practical 01: User-Item Matrix and Sparsity

Learning objectives:
- Build a user-item rating matrix from interactions.
- Measure density and sparsity.
- Understand why collaborative filtering needs overlap.
- Visualize rating distributions and missing values.

Slide connection: collaborative filtering motivation, memory-based CF overview, and user-item rating matrix.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Memory-based CF keeps the interaction matrix and uses it directly at recommendation time.


In [ ]:
n_users, n_items = rating_matrix.shape
n_known = rating_matrix.notna().sum().sum()
density = n_known / (n_users * n_items)

summary = pd.DataFrame({
    "measure": ["users", "items", "known ratings", "possible ratings", "density", "sparsity"],
    "value": [n_users, n_items, n_known, n_users * n_items, round(density, 3), round(1 - density, 3)],
})
summary


Missing values are not zero ratings. They mean unknown preference.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(rating_matrix.notna(), aspect="auto", cmap="Greens")
ax.set_xticks(range(len(rating_matrix.columns)), rating_matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(len(rating_matrix.index)), rating_matrix.index)
ax.set_title("Known ratings in the user-item matrix")
ax.set_xlabel("Movie")
ax.set_ylabel("User")
fig.tight_layout()
fig


The number of co-rated items determines how trustworthy a user-user comparison can be.


In [ ]:
users = rating_matrix.index
overlap = pd.DataFrame(index=users, columns=users, dtype=int)
for u in users:
    for v in users:
        overlap.loc[u, v] = rating_matrix.loc[[u, v]].notna().all(axis=0).sum()
overlap


In [ ]:
ratings_named.groupby("title")["rating"].agg(["count", "mean"]).sort_values(["count", "mean"], ascending=False)


# Challenges

### Challenge 1 — Add a Sparse New User

**Goal:**
Investigate how a user with very few ratings changes the user-item matrix and co-rated-item overlap.

**What to do:**

1. Add a new user with only one movie rating to `ratings_named`.
2. Rebuild `rating_matrix` from the updated data.
3. Rerun the sparsity and overlap cells.
4. Compare the new user's overlap with the existing users.


In [ ]:
# Challenge 1
# Modify or extend the code as described above.

# Write your code below:


### Your observations

What happened to matrix sparsity? How much overlap did the new user have with other users? Why does this make memory-based CF difficult?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 2 — Add a Rating for a Rare Movie

**Goal:**
Investigate how adding one extra interaction changes item popularity and available evidence.

**What to do:**

1. Choose a movie with a low `ratings_count` in the popularity table.
2. Add one new rating for that movie.
3. Rebuild the matrix and rerun the popularity summary.
4. Compare the movie's count and mean rating before and after the change.


In [ ]:
# Challenge 2
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Which movie changed most? Did one extra rating make the item easier to use in collaborative filtering? Why?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 3 — Concept Check: Missing Ratings

This challenge requires **no programming**.

A user has not rated `Titanic`. The rating matrix shows this as a missing value.

Explain in your own words why this missing value should not be treated as a real rating of zero.

### Your explanation

> ................................................................................
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
